# Half-cell phase — repro first, actual symbol second
CPU/high RAM. Fresh pinned checkout. Diagnostic only: no production seal, inverse, B0 or window15. One execution; preserve first error.

In [ ]:
import hashlib, pathlib, subprocess, datetime, psutil, urllib.request
SOURCE_SHA = "7320d1bab92f0bcb18d8072dd0d2fa13b1aea204"
RUNNER_REV = "neumann-half-cell-phase-diagnostic-v1"
RUNNER_SHA = "0505a1567cd48a5a9838e50103be80108324d171"
RUNNER_HASH = "ecd76d0c05609aa45a4dc1c69e33ae9fa0dfb8600283760464c8d547697a42b8"
p = pathlib.Path("/content/launch-neumann-half-cell-phase-v1.py")
log = pathlib.Path("/content/launch-neumann-half-cell-phase-v1.log")
assert psutil.virtual_memory().total / 2**30 >= 40, "HIGH_RAM_REQUIRED"
assert not pathlib.Path("/dev/nvidia0").exists(), "GPU_NOT_AUTHORIZED"
assert not p.exists() and not log.exists(), "ALREADY_STARTED_NO_REEXECUTION"
url = "https://raw.githubusercontent.com/lluiseriksson/THE-ERIKSSON-PROGRAMME/" + RUNNER_SHA + "/scripts/colab_neumann_half_cell_phase_diagnostic.py"
blob = urllib.request.urlopen(url, timeout=60).read()
assert hashlib.sha256(blob).hexdigest() == RUNNER_HASH, "TRANSPORT_HASH"
with p.open("xb") as f: f.write(blob)
print("SOURCE_SHA=" + SOURCE_SHA + " RUNNER_REV=" + RUNNER_REV, flush=True)
print("HASH_GATE=PASS START_UTC=" + datetime.datetime.now(datetime.timezone.utc).isoformat(), flush=True)
with log.open("xb") as f:
    child = subprocess.Popen(["/usr/bin/python3", str(p)], stdout=f, stderr=subprocess.STDOUT)
    print("LAUNCH_PID=" + str(child.pid), flush=True)
    exit_code = child.wait()
print(log.read_text(errors="replace")[-20000:], flush=True)
print("LAUNCHER_EXIT=" + str(exit_code), flush=True)
archive = pathlib.Path("/content/hrpoly-neumann-half-cell-phase-diagnostic-v1-evidence.tar.gz")
if archive.is_file():
    print("ARCHIVE_SHA256=" + hashlib.sha256(archive.read_bytes()).hexdigest(), flush=True)
    from google.colab import files
    files.download(str(archive))
if exit_code:
    raise RuntimeError("First error preserved; do not reexecute.")

